In [26]:
import pickle
import pandas as pd
import numpy as np

In [7]:
model_path="C:\\mlops_boot_camp\\mlops-zoomcamp\\homework_3_orchestration\\models\\linear_regression_model.pkl"
vectorizer_path="C:\\mlops_boot_camp\\mlops-zoomcamp\\homework_3_orchestration\\models\\dv.pkl"
data_path="C:\\mlops_boot_camp\\mlops-zoomcamp\\homework_3_orchestration\\data\\yellow_tripdata_2023-03.parquet"

In [6]:
with open(model_path, 'rb') as f_in:
    model = pickle.load(f_in)

In [8]:
with open(vectorizer_path, 'rb') as f_in:
    vectorizer = pickle.load(f_in)

In [9]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [12]:
df = read_data(data_path)

In [13]:
dicts = df[categorical].to_dict(orient='records')
X_val = vectorizer.transform(dicts)
y_pred = model.predict(X_val)

## What's the standard deviation of the predicted duration for this dataset?

In [14]:
import numpy as np

std_pred = np.std(y_pred)
print(std_pred)

6.767451713844294


## Q2. Preparing the output

In [16]:
year=2023
month=3
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [18]:
df['predictions']=y_pred

In [19]:
df_result=df[['ride_id','predictions']]

In [21]:
output_file="C:\\mlops_boot_camp\\mlops-zoomcamp\\homework_4\\data\\homework_4_results.parquet"
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [22]:
import os

file_path = r"C:\mlops_boot_camp\mlops-zoomcamp\homework_4\data\homework_4_results.parquet"

size_bytes = os.path.getsize(file_path)

print(f"File size in bytes: {size_bytes}")
print(f"File size in KB: {size_bytes / 1024:.2f}")
print(f"File size in MB: {size_bytes / (1024 * 1024):.2f}")

File size in bytes: 68557923
File size in KB: 66951.10
File size in MB: 65.38


## Q3. Creating the scoring script

### Answer : jupyter nbconvert --to script homework_4.ipynb

## Q4. Virtual environment

In [24]:
json_path="C:\\mlops_boot_camp\\mlops-zoomcamp\\Pipfile.lock"

In [25]:
import json

with open(json_path, "r") as f:
    lock_data = json.load(f)

print(lock_data["default"]["scikit-learn"]["hashes"][0])


sha256:051075bda8b7aab87b1906ab3d4740a1e1224a19d7b3781a576736edc94e76aa


## Q5. Parametrize the scriptc

### What's the mean predicted duration?

### Answer : 6.877047353419602